# Step 2 — Product profile (spec.md + exploded view → one JSON)

Every later stage (regulation filter, section filter, requirement extraction) reads **one product profile**.

Two steps produce it:
1. **Deterministic** — parse the headings of `spec.md` (Product Type, Vehicle Category, Function, Main Parts, Materials). No model; always works; the fallback on Sunday if the API is down.
2. **Model + vision** — send spec text + drawing to Claude and ask it to confirm and enrich: parts visible in the drawing that the spec does not name, the term the regulations use for each part (`regulated_function`), electronics, light sources, and a quoted piece of **evidence** per part (shown in the demo).

Output: `work/profiles/<case_id>.json`. Needs `ANTHROPIC_API_KEY` in the environment (or in `certivity/.env`) for step 2.

In [1]:
from pathlib import Path
import json, re, os, base64, sys
ROOT = Path.cwd() if (Path.cwd() / "package").exists() else Path.home() / "Desktop/certivity"
PKG, WORK = ROOT / "package", ROOT / "work"
(WORK / "profiles").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))

# load ANTHROPIC_API_KEY from certivity/.env if the shell did not export it
if not os.environ.get("ANTHROPIC_API_KEY") and (ROOT / ".env").exists():
    for line in (ROOT / ".env").read_text().splitlines():
        if line.startswith("ANTHROPIC_API_KEY=") and line.split("=", 1)[1].strip():
            os.environ["ANTHROPIC_API_KEY"] = line.split("=", 1)[1].strip().strip('"')
print("ROOT:", ROOT, "| API key:", "set" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING")

ROOT: /Users/arnavsingh/Desktop/certivity | API key: set


In [2]:
%%writefile work/profile.py
"""Product profile: deterministic parse of spec.md + optional Claude vision enrichment (generated from the notebook)."""
from __future__ import annotations
import base64, json, re
from pathlib import Path

MODEL = "claude-opus-5-5"

# ------------------------------------------------------------------ 1. deterministic ------------------------------
def parse_spec_md(md: str) -> dict:
    """Split a spec markdown into its '## ' sections and pull the fields we rely on."""
    sections, cur = {}, None
    for line in md.splitlines():
        if (m := re.match(r"^#+\s*(.+?)\s*$", line)):
            cur = m.group(1).strip().lower(); sections[cur] = []
        elif cur is not None:
            sections[cur].append(line)
    sec = {k: "\n".join(v).strip() for k, v in sections.items()}

    def bullets(text):  return [re.sub(r"^[-*]\s*", "", l).strip() for l in text.splitlines() if re.match(r"^\s*[-*]\s+", l)]
    def first(text):    return next((l.strip() for l in text.splitlines() if l.strip()), "")
    def table(text):
        rows = [[c.strip() for c in l.strip().strip("|").split("|")] for l in text.splitlines() if l.strip().startswith("|")]
        rows = [r for r in rows if not all(re.fullmatch(r"-+", c) for c in r)]
        return {r[0]: r[1] for r in rows[1:] if len(r) >= 2} if rows else {}

    product_line = first(sec.get("product type", ""))
    cat_line = first(sec.get("vehicle category", ""))
    title = first(md).lstrip("# ").strip()
    return {
        "product": re.split(r"\s+[—–-]\s+", product_line)[0].strip() or re.sub(r"^spec:\s*", "", title, flags=re.I),
        "product_description": product_line,
        "vehicle_category": (re.match(r"\s*([A-Z]\d?)\b", cat_line).group(1) if re.match(r"\s*([A-Z]\d?)\b", cat_line) else None),
        "vehicle_category_note": cat_line,
        "functions": bullets(sec.get("function", "")),
        "parts": [{"part_id": f"P-{i+1:02d}", "name": p} for i, p in enumerate(bullets(sec.get("main parts", "")))],
        "materials": table(sec.get("materials", "")),
        "spec_sections": sorted(sec),
    }


# ------------------------------------------------------------------ 2. model + vision ------------------------------
PROFILE_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "required": ["product", "vehicle_category", "mounting", "parts", "has_electronics", "electronic_subassemblies",
                 "light_sources", "regulated_functions", "search_terms", "drawing_observations"],
    "properties": {
        "product": {"type": "string"},
        "vehicle_category": {"type": "string", "description": "UNECE category, e.g. M1, N1, L3"},
        "mounting": {"type": "string", "description": "where on the vehicle, e.g. 'rear exterior', 'front exterior'"},
        "parts": {"type": "array", "items": {"type": "object", "additionalProperties": False,
            "required": ["part_id", "name", "function", "regulated_function", "material", "evidence_spec", "evidence_drawing"],
            "properties": {
                "part_id": {"type": "string"},
                "name": {"type": "string"},
                "function": {"type": "string", "description": "what the part does, in engineering terms"},
                "regulated_function": {"type": ["string", "null"], "description": "the exact term UN Regulations use for this part/function (e.g. 'rear position lamp', 'stop lamp', 'retro-reflector Class IA', 'electrical/electronic sub-assembly (ESA)', 'front fog lamp', 'external projection'); null if none"},
                "material": {"type": ["string", "null"]},
                "evidence_spec": {"type": ["string", "null"], "description": "verbatim quote from the spec that supports this part, or null"},
                "evidence_drawing": {"type": ["string", "null"], "description": "what in the drawing shows this part, or null"}}}},
        "has_electronics": {"type": "boolean"},
        "electronic_subassemblies": {"type": "array", "items": {"type": "string"}},
        "light_sources": {"type": ["string", "null"], "description": "e.g. 'non-replaceable LED modules', 'replaceable filament lamp category P21W', or null"},
        "regulated_functions": {"type": "array", "items": {"type": "string"}, "description": "deduplicated list of all regulated_function values"},
        "search_terms": {"type": "array", "items": {"type": "string"}, "description": "terms to search the regulations for: part names, functions, and their synonyms as the regulations phrase them"},
        "drawing_observations": {"type": "array", "items": {"type": "string"}, "description": "what the exploded view shows, as short factual notes"},
    },
}

SYSTEM = """You are a vehicle type-approval engineer. You receive a short technical specification of a vehicle component and,
usually, an exploded-view drawing of it. Produce a product profile that a regulatory engineer would use to decide which
UN Regulations (UNECE, 1958 Agreement) and which of their sections apply to the component.

Rules:
- Use the spec as the primary source and the drawing to confirm, refine and add parts the spec does not name.
- Name regulated functions with the exact vocabulary the UN Regulations use (e.g. "rear position lamp", "stop lamp",
  "reversing lamp", "retro-reflector", "front fog lamp", "daytime running lamp", "direction indicator", "electrical/electronic
  sub-assembly (ESA)", "external projection", "bumper", "protective device"). If a part has no regulated function, say null.
- Any electronics (LED driver, PCB, sensors, control gear) make the component an ESA for electromagnetic compatibility purposes.
- Light sources: say whether they are replaceable (then the light-source regulations apply) or non-replaceable modules.
- Quote evidence verbatim from the spec; describe drawing evidence factually. Do not invent parts that neither source supports.
- Keep the vehicle category exactly as stated in the spec."""


def _image_block(path: Path) -> dict:
    media = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    return {"type": "image", "source": {"type": "base64", "media_type": media,
                                        "data": base64.standard_b64encode(path.read_bytes()).decode()}}


def enrich_with_model(spec_md: str, deterministic: dict, image_paths: list[Path], client=None) -> dict:
    import anthropic
    client = client or anthropic.Anthropic()
    content = [_image_block(p) for p in image_paths]
    content.append({"type": "text", "text":
        "## Specification (markdown)\n\n" + spec_md +
        "\n\n## Fields already parsed from the specification (confirm or correct)\n\n" + json.dumps(deterministic, indent=1) +
        ("\n\nThe image(s) above are the exploded view(s) of the component." if image_paths else "\n\nNo drawing was provided.")})
    with client.messages.stream(
        model=MODEL, max_tokens=16000,
        system=SYSTEM,
        output_config={"effort": "high", "format": {"type": "json_schema", "schema": PROFILE_SCHEMA}},
        messages=[{"role": "user", "content": content}],
    ) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == "refusal":
        raise RuntimeError(f"model refused: {msg.stop_details}")
    text = next(b.text for b in msg.content if b.type == "text")
    out = json.loads(text)
    out["_model"] = msg.model
    out["_usage"] = {"input": msg.usage.input_tokens, "output": msg.usage.output_tokens}
    return out


# ------------------------------------------------------------------ 3. entry point ---------------------------------
def build_profile(case_dir: Path, use_model: bool = True, client=None) -> dict:
    case_dir = Path(case_dir)
    spec_path = next(iter(sorted(case_dir.glob("*.md"))), None)
    images = sorted(p for p in case_dir.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
    spec_md = spec_path.read_text() if spec_path else ""
    det = parse_spec_md(spec_md) if spec_md else {"product": None, "parts": [], "functions": [], "materials": {}}
    profile = {"case_id": case_dir.name, "spec_file": spec_path.name if spec_path else None,
               "image_files": [p.name for p in images], "deterministic": det}
    if use_model:
        profile["model"] = enrich_with_model(spec_md, det, images, client)
    return profile


Overwriting work/profile.py


In [3]:
import importlib, profile as _p; importlib.reload(_p)
case = PKG / "cases/dev-rear-lamp"
det = _p.parse_spec_md((case / "spec.md").read_text())
print(json.dumps(det, indent=1))

{
 "product": "Rear lamp",
 "product_description": "Rear lamp  \u2014 exterior lighting & signalling module combining multiple light functions, lens, and retro-reflector in a single housing.",
 "vehicle_category": "M1",
 "vehicle_category_note": "M1 \u2014 passenger car. UNECE lighting regulations referenced apply across M and N categories.",
 "functions": [
  "Rear position signalling (presence/visibility)",
  "Braking indication (stop lamp)",
  "Reversing illumination",
  "Passive conspicuity (retro-reflector)",
  "Light transmission & color definition (outer lens)"
 ],
 "parts": [
  {
   "part_id": "P-01",
   "name": "Rear position lamp"
  },
  {
   "part_id": "P-02",
   "name": "Stop lamp"
  },
  {
   "part_id": "P-03",
   "name": "Reversing lamp"
  },
  {
   "part_id": "P-04",
   "name": "Outer lens"
  },
  {
   "part_id": "P-05",
   "name": "Retro-reflector"
  }
 ],
 "materials": {
  "Outer lens": "PMMA (red-tinted, UV-stabilised)",
  "Housing": "PP or ASA/PC",
  "Reflector surfa

In [4]:
# --- step 2: model + vision (runs only when the API key is available) ---
if os.environ.get("ANTHROPIC_API_KEY"):
    profile = _p.build_profile(case, use_model=True)
    out = WORK / "profiles" / f"{case.name}.json"
    out.write_text(json.dumps(profile, indent=1, ensure_ascii=False))
    m = profile["model"]
    print(f"model={m['_model']} tokens={m['_usage']}\n")
    print("product:", m["product"], "| category:", m["vehicle_category"], "| mounting:", m["mounting"])
    print("electronics:", m["has_electronics"], m["electronic_subassemblies"], "| light sources:", m["light_sources"])
    print("regulated functions:", m["regulated_functions"])
    print("\nparts:")
    for p in m["parts"]:
        print(f"  {p['part_id']} {p['name']:<28} -> {p['regulated_function']}")
        print(f"       spec: {p['evidence_spec']!r}")
        print(f"       drawing: {p['evidence_drawing']!r}")
    print("\ndrawing observations:", *m["drawing_observations"], sep="\n  - ")
    print("\nsaved:", out)
else:
    print("ANTHROPIC_API_KEY not set: skipped the model step. Export it (or put it in certivity/.env) and re-run this cell.")

model=claude-opus-5-5 tokens={'input': 7498, 'output': 3072}

product: Rear lamp | category: M1 | mounting: rear exterior
electronics: True ['LED modules on PCB (light source board with electronic components/connector)'] | light sources: LED modules on PCB; replaceability not stated in spec - appear to be PCB-mounted non-replaceable LED light sources/modules (to be confirmed)
regulated functions: ['rear position lamp', 'stop lamp', 'reversing lamp', 'outer lens (lens of signalling lamp; colour of emitted light - red)', 'retro-reflector', 'electrical/electronic sub-assembly (ESA)']

parts:
  P-01 Rear position lamp           -> rear position lamp
       spec: 'Rear position signalling (presence/visibility)'
       drawing: "Reflector callout text: 'rear position lamp: 4 to 17 cd on axis'"
  P-02 Stop lamp                    -> stop lamp
       spec: 'Braking indication (stop lamp)'
       drawing: "LED modules on PCB callout text: 'stop lamp: 60 to 185 cd on axis'"
  P-03 Reversing lamp